In [ ]:
import numpy as np
import cv2
from collections import deque

def setValues(x):
    pass

# Create trackbars for color detection
cv2.namedWindow("Color detectors", cv2.WINDOW_NORMAL)
cv2.createTrackbar("Upper Hue", "Color detectors", 153, 180, setValues)
cv2.createTrackbar("Upper Saturation", "Color detectors", 255, 255, setValues)
cv2.createTrackbar("Upper Value", "Color detectors", 255, 255, setValues)
cv2.createTrackbar("Lower Hue", "Color detectors", 64, 180, setValues)
cv2.createTrackbar("Lower Saturation", "Color detectors", 72, 255, setValues)
cv2.createTrackbar("Lower Value", "Color detectors", 49, 255, setValues)

# Create deques for storing points
bpoints = [deque(maxlen=1024)]
gpoints = [deque(maxlen=1024)]
rpoints = [deque(maxlen=1024)]
ypoints = [deque(maxlen=1024)]

# Set up colors and paint window
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (0, 255, 255)]
paintWindow = np.zeros((471, 636, 3)) + 255

# Draw color buttons
def draw_color_buttons(window):
    cv2.rectangle(window, (40, 1), (140, 65), (0, 0, 0), 2)
    for i, color in enumerate(colors):
        cv2.rectangle(window, (160 + (i * 115), 1), (255 + (i * 115), 65), color, -1)
        cv2.putText(window, ["CLEAR", "BLUE", "GREEN", "RED", "YELLOW"][i],
                    (185 + (i * 115), 33), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

draw_color_buttons(paintWindow)
cv2.namedWindow('Paint', cv2.WINDOW_NORMAL)

# Initialize video capture
cap = cv2.VideoCapture(0)  # Use 0 for the default camera; adjust if needed

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break

    frame = cv2.flip(frame, 1)
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Get trackbar positions
    u_hue = cv2.getTrackbarPos("Upper Hue", "Color detectors")
    u_saturation = cv2.getTrackbarPos("Upper Saturation", "Color detectors")
    u_value = cv2.getTrackbarPos("Upper Value", "Color detectors")
    l_hue = cv2.getTrackbarPos("Lower Hue", "Color detectors")
    l_saturation = cv2.getTrackbarPos("Lower Saturation", "Color detectors")
    l_value = cv2.getTrackbarPos("Lower Value", "Color detectors")
    
    Upper_hsv = np.array([u_hue, u_saturation, u_value])
    Lower_hsv = np.array([l_hue, l_saturation, l_value])

    # Draw buttons on the frame
    draw_color_buttons(frame)

    # Create mask
    Mask = cv2.inRange(hsv, Lower_hsv, Upper_hsv)
    Mask = cv2.erode(Mask, np.ones((5, 5), np.uint8), iterations=1)
    Mask = cv2.morphologyEx(Mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    Mask = cv2.dilate(Mask, np.ones((5, 5), np.uint8), iterations=1)

    cnts, _ = cv2.findContours(Mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    center = None

    if len(cnts) > 0:
        cnt = sorted(cnts, key=cv2.contourArea, reverse=True)[0]
        ((x, y), radius) = cv2.minEnclosingCircle(cnt)
        if radius > 10:  # Only consider large enough circles
            cv2.circle(frame, (int(x), int(y)), int(radius), (0, 255, 255), 2)
            M = cv2.moments(cnt)
            center = (int(M['m10'] / M['m00']), int(M['m01'] / M['m00']))

            if center[1] <= 65:
                if 40 <= center[0] <= 140:  # Clear button
                    bpoints = [deque(maxlen=512)]
                    gpoints = [deque(maxlen=512)]
                    rpoints = [deque(maxlen=512)]
                    ypoints = [deque(maxlen=512)]
                else:
                    colorIndex = (center[0] - 160) // 115
                    if 0 <= colorIndex < len(colors):
                        # Logic for handling selected color
                        pass

    # Show the frames
    cv2.imshow("Paint", paintWindow)
    cv2.imshow("Frame", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


2024-09-30 23:51:14.661 python[39416:1920277] +[IMKClient subclass]: chose IMKClient_Legacy
2024-09-30 23:51:14.661 python[39416:1920277] +[IMKInputSession subclass]: chose IMKInputSession_Legacy
